# Training Outfit Transformer with Image->Text Retrieval
- Loads pre-prepared cropped dataset images (512-dim)
- Loads text FAISS index (built by 3_build_text_faiss.py)
- Builds 3 versions of 1024-dim embeddings:
  - A: Top-1 text retrieval
  - B: Avg top-5 text retrieval
  - C: Image duplicated
- Trains 3 models using these embeddings natively (no image_only adapter needed)
- Evaluates on TEST split and compares to baseline

In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 85.4 MB/s eta 0:00:00


In [2]:
import random
import numpy as np
import torch
import os
import glob
import json
import faiss
import time
from pathlib import Path
from tqdm.auto import tqdm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("✅ WANDB_API_KEY loaded")
except Exception:
    print("⚠ WandB will run in offline mode")
    os.environ["WANDB_MODE"] = "offline"

wandb.init(
    project="fitfinder-outfit-transformer-retrieval",
    config={
        "model_type": "clip",
        "polyvore_split": "disjoint",
        "seed": seed,
        "n_epochs_compatibility": 10,
        "n_epochs_complementary": 10,
        "lr": 1e-3,
    },
    resume="allow",
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: moh000abdo (moh000abdo-alexandria-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ WANDB_API_KEY loaded


wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260523_211103-hp40o9ns
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run noble-wildflower-4
wandb: ⭐️ View project at https://wandb.ai/moh000abdo-alexandria-university/fitfinder-outfit-transformer-retrieval
wandb: 🚀 View run at https://wandb.ai/moh000abdo-alexandria-university/fitfinder-outfit-transformer-retrieval/runs/hp40o9ns


## Clone Repo & Verify Data

In [4]:
import sys
import os
from pathlib import Path

In [5]:
# Assume we run inside outfit-transformer or clone it
if not Path("outfit-transformer").exists() and not Path("src/run").exists():
    os.system("git clone https://github.com/bigohofone/outfit-transformer.git")
    os.chdir("outfit-transformer")
elif Path("outfit-transformer").exists():
    os.chdir("outfit-transformer")

sys.path.insert(0, '.')

Cloning into 'outfit-transformer'...


In [6]:
DATASET_INPUTS = [
    Path("/kaggle/input/polyvore-disjoint-cropped-garments"),
    Path("/kaggle/input/datasets/mabdelmoneim/polyvore-disjoint-cropped-garments"),
    Path("/kaggle/working/cropped_polyvore"),
    Path("../cropped_polyvore"),  # Local fallback
]

PREPARED_DATA = None
for dataset_path in DATASET_INPUTS:
    if dataset_path.exists():
        PREPARED_DATA = dataset_path
        print(f"✅ Using Image/Embedding Dataset: {PREPARED_DATA}")
        break

if PREPARED_DATA is None:
    raise FileNotFoundError("Prepared cropped dataset not found! Ensure 'polyvore-disjoint-cropped-garments' is attached to the notebook.")

embeddings_dir = PREPARED_DATA / "embeddings_normalized"
if not embeddings_dir.exists():
    embeddings_dir = PREPARED_DATA / "embeddings"

✅ Using Image/Embedding Dataset: /kaggle/input/datasets/mabdelmoneim/polyvore-disjoint-cropped-garments


In [7]:
FAISS_INPUTS = [
    Path("/kaggle/input/polyvore-text-faiss-embeddings"), # The new Kaggle dataset
    Path("/kaggle/input/datasets/mabdelmoneim/polyvore-text-faiss-embeddings"),
    Path("/kaggle/working/text_faiss_data"),              # If generated in the current live session
    Path("../text_faiss_data"),                           # Local fallback
]

FAISS_DATA_DIR = None
for faiss_path in FAISS_INPUTS:
    if faiss_path.exists():
        FAISS_DATA_DIR = faiss_path
        print(f"✅ Using FAISS Dataset: {FAISS_DATA_DIR}")
        break

if FAISS_DATA_DIR is None:
    raise FileNotFoundError("FAISS data directory not found! Ensure your new 'polyvore-text-faiss-embeddings' dataset is attached to the notebook.")
    

✅ Using FAISS Dataset: /kaggle/input/datasets/mabdelmoneim/polyvore-text-faiss-embeddings


## Patch Source Files (No `image_only` hack, native 1024-dim support)

In [8]:
import re

### 1. Polyvore dataset - Support loading from NPZ and safe loading

In [9]:
with open('src/data/datasets/polyvore.py', 'r') as f:
    content = f.read()

# Fix index mapping
content = content.replace(
    'metadata_idx2item_id = {\n    j: i for i, j in tqdm(enumerate(item_id2metadata_idx))\n}',
    'metadata_idx2item_id = {\n    i: j for i, j in enumerate(item_id2metadata_idx.keys())\n}'
)

if 'def load_metadata' not in content:
    content += "\n\n\ndef load_metadata(dataset_dir=None):\n    return globals().get('metadata')\n"

# Patch load_embedding_dict to support loading directly from an NPZ file
if 'npz_path=None' not in content:
    content = content.replace(
        "def load_embedding_dict(dataset_dir):",
        "def load_embedding_dict(dataset_dir, npz_path=None):\n"
        "    if npz_path and os.path.exists(npz_path):\n"
        "        print(f'Loading embeddings from NPZ: {npz_path}')\n"
        "        data = np.load(npz_path)\n"
        "        return {k: data[k] for k in data.files}\n"
    )

# Add safe_load_item function
if 'def safe_load_item' not in content:
    content = content.replace(
        'def load_item(',
        'def safe_load_item(item_id, id_converter, embedding_dict):\n'
        '    """Safely load item, skipping empty or missing IDs."""\n'
        '    if not item_id or item_id not in id_converter:\n'
        '        return None\n'
        '    return load_item(id_converter[item_id], embedding_dict)\n\n\n'
        'def load_item('
    )

content = content.replace(
    'load_item(self.id_converter[item_id], self.embedding_dict)',
    'safe_load_item(item_id, self.id_converter, self.embedding_dict)'
)
content = content.replace(
    'query=FashionCompatibilityQuery(outfit=outfit)',
    'query=FashionCompatibilityQuery(outfit=[x for x in outfit if x is not None])'
)
content = content.replace(
    'return FashionFillInTheBlankData(\n'
    '            query=FashionComplementaryQuery(outfit=outfit, category=candidates[label].category),\n'
    '            label=label,\n'
    '            candidates=candidates\n'
    '        )',
    'candidates = [x for x in candidates if x is not None]\n'
    '        outfit = [x for x in outfit if x is not None]\n'
    '        if not candidates or label >= len(candidates):\n'
    '            import random\n'
    '            return self.__getitem__(random.randint(0, len(self.data) - 1))\n'
    '        return FashionFillInTheBlankData(\n'
    '            query=FashionComplementaryQuery(outfit=outfit, category=candidates[label].category),\n'
    '            label=label,\n'
    '            candidates=candidates\n'
    '        )'
)

with open('src/data/datasets/polyvore.py', 'w') as f:
    f.write(content)
print("✅ Patched polyvore.py for safe loading and NPZ support")


✅ Patched polyvore.py for safe loading and NPZ support


In [10]:
# Download base checkpoints if not present
os.system("mkdir -p checkpoints")
if not Path("checkpoints/compatibillity_clip_best.pth").exists():
    os.system("gdown --id 1mzNqGBmd8UjVJjKwVa5GdGYHKutZKSSi -O checkpoints.zip && unzip -q -o checkpoints.zip -d ./checkpoints && rm -f checkpoints.zip")


# Clear previously patched load.py if needed, we just use the original or a simple rewrite
# (OutfitTransformer doesn't need image_only or img_proj since our embeddings are 1024)
load_py_content = '''import torch
from .outfit_transformer import OutfitTransformerConfig, OutfitTransformer
from .outfit_clip_transformer import OutfitCLIPTransformerConfig, OutfitCLIPTransformer

def load_model(model_type, checkpoint=None, image_only=False, **cfg_kwargs):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    state_dict = None
    if checkpoint:
        state_dict = torch.load(checkpoint, map_location=device)
        cfg = state_dict.get('config', {})
        model_state_dict = state_dict.get('model', {})
    else:
        cfg = cfg_kwargs
        model_state_dict = None

    if 'image_only' in cfg:
        cfg['image_only'] = image_only

    if model_type == 'original':
        model = OutfitTransformer(OutfitTransformerConfig(**cfg))
    elif model_type == 'clip':
        model = OutfitCLIPTransformer(OutfitCLIPTransformerConfig(**cfg))

    model.to(device)

    if model_state_dict:
        new_state_dict = {k.replace("module.", ""): v for k, v in model_state_dict.items()}
        # For baseline, we might need strict=False if it has adapter weights
        model.load_state_dict(new_state_dict, strict=False)
    
    return model
'''
with open('src/models/load.py', 'w') as f:
    f.write(load_py_content)
print("✅ Rewritten load.py")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1mzNqGBmd8UjVJjKwVa5GdGYHKutZKSSi
From (redirected): https://drive.google.com/uc?id=1mzNqGBmd8UjVJjKwVa5GdGYHKutZKSSi&confirm=t&uuid=7ea0eb64-1389-41ee-b8f9-5b886b616508
To: /kaggle/working/outfit-transformer/checkpoints.zip
100%|██████████| 1.15G/1.15G [00:13<00:00, 88.0MB/s]


✅ Rewritten load.py


### Filtered FAISS Index for Training

In [11]:
print("\n" + "="*50)
print("Loading Text FAISS Data...")

with open(FAISS_DATA_DIR / "text_faiss_item_ids.json") as f:
    all_faiss_item_ids = json.load(f)

with open(FAISS_DATA_DIR / "text_faiss_split_info.json") as f:
    split_info = json.load(f)

all_text_embeddings = np.load(FAISS_DATA_DIR / "text_embeddings_all.npy")
full_faiss_index = faiss.read_index(str(FAISS_DATA_DIR / "text_faiss_index.bin"))

# Build filtered training index
print("Building filtered FAISS index for training (excluding test & unassigned)...")
train_val_indices = []
train_val_item_ids = []

for i, item_id in enumerate(all_faiss_item_ids):
    splits = split_info.get(item_id, [])
    # Include if it has 'train' or 'validation' but NOT 'test' and NOT 'unassigned'
    if ("train" in splits or "validation" in splits) and ("test" not in splits) and ("unassigned" not in splits):
        train_val_indices.append(i)
        train_val_item_ids.append(item_id)

train_val_embeddings = all_text_embeddings[train_val_indices]
train_val_faiss_index = faiss.IndexFlatIP(512)
train_val_faiss_index.add(train_val_embeddings)
print(f"Filtered Index size: {train_val_faiss_index.ntotal} (vs Full: {full_faiss_index.ntotal})")


Loading Text FAISS Data...
Building filtered FAISS index for training (excluding test & unassigned)...
Filtered Index size: 82750 (vs Full: 251008)


## Pre-build Combined Embeddings (Methods A, B, C)

In [12]:
NPZ_DIR = Path("/kaggle/working/combined_embeddings")
if not NPZ_DIR.parent.exists():
    NPZ_DIR = Path("../combined_embeddings")
NPZ_DIR.mkdir(parents=True, exist_ok=True)

# Pre-compute all three
npz_A = NPZ_DIR / "method_A.npz"
npz_B = NPZ_DIR / "method_B.npz"
npz_C = NPZ_DIR / "method_C.npz"


In [13]:
def build_combined_embeddings_fast(method, out_npz_path, batch_size=10000):
    if out_npz_path.exists():
        print(f"✅ Method {method} embeddings already exist at {out_npz_path}, skipping.")
        return

    print(f"\nBuilding combined embeddings for Method {method} (batched)...")

    # 1. Collect all image files and split by index type
    img_files = list(embeddings_dir.glob("*.npy"))
    test_unassigned_ids = []
    train_val_ids = []
    for p in img_files:
        item_id = p.stem
        splits = split_info.get(item_id, [])
        if "test" in splits or "unassigned" in splits:
            test_unassigned_ids.append(item_id)
        else:
            train_val_ids.append(item_id)

    # 2. Helper: load embeddings for a list of IDs and return (ids, matrix)
    def load_embeddings(id_list):
        if not id_list:
            return [], np.empty((0, 512))
        # Load and stack
        embs = [np.load(embeddings_dir / f"{i}.npy") for i in id_list]
        return id_list, np.stack(embs)

    # 3. Process each group, possibly in batches to control memory
    combined = {}
    groups = [
        (test_unassigned_ids, full_faiss_index, all_text_embeddings, "test/unassigned"),
        (train_val_ids, train_val_faiss_index, train_val_embeddings, "train/val")
    ]

    for ids, index, text_matrix, desc in groups:
        if not ids:
            continue
        print(f"  Processing {len(ids)} items from {desc}...")
        # If many items, process in sub‑batches to keep memory low
        for start in tqdm(range(0, len(ids), batch_size), desc=f"Method {method} {desc}"):
            batch_ids = ids[start:start+batch_size]
            # Load image embeddings for this batch
            img_batch = np.stack([np.load(embeddings_dir / f"{i}.npy") for i in batch_ids])

            if method == "A":
                _, I = index.search(img_batch, k=1)           # (batch, 1)
                text_embs = text_matrix[I[:, 0]]              # (batch, 512)
                combined_batch = np.concatenate([img_batch, text_embs], axis=1)

            elif method == "B":
                _, I = index.search(img_batch, k=5)           # (batch, 5)
                # average top‑5 and L2 normalise each row
                avg_text = text_matrix[I].mean(axis=1)        # (batch, 512)
                norm = np.linalg.norm(avg_text, axis=1, keepdims=True)
                norm[norm == 0] = 1.0   # avoid division by zero
                avg_text = avg_text / norm
                combined_batch = np.concatenate([img_batch, avg_text], axis=1)

            elif method == "C":
                combined_batch = np.concatenate([img_batch, img_batch], axis=1)

            # Store in dictionary
            for i, item_id in enumerate(batch_ids):
                combined[item_id] = combined_batch[i]

    # 4. Save
    np.savez(out_npz_path, **combined)
    print(f"💾 Saved {len(combined)} embeddings to {out_npz_path}")

In [14]:
build_combined_embeddings_fast("A", npz_A)


Building combined embeddings for Method A (batched)...
  Processing 168258 items from test/unassigned...


Method A test/unassigned:   0%|          | 0/17 [00:00<?, ?it/s]

  Processing 82750 items from train/val...


Method A train/val:   0%|          | 0/9 [00:00<?, ?it/s]

💾 Saved 251008 embeddings to /kaggle/working/combined_embeddings/method_A.npz


In [15]:
build_combined_embeddings_fast("B", npz_B)


Building combined embeddings for Method B (batched)...
  Processing 168258 items from test/unassigned...


Method B test/unassigned:   0%|          | 0/17 [00:00<?, ?it/s]

  Processing 82750 items from train/val...


Method B train/val:   0%|          | 0/9 [00:00<?, ?it/s]

💾 Saved 251008 embeddings to /kaggle/working/combined_embeddings/method_B.npz


In [16]:
build_combined_embeddings_fast("C", npz_C)


Building combined embeddings for Method C (batched)...
  Processing 168258 items from test/unassigned...


Method C test/unassigned:   0%|          | 0/17 [00:00<?, ?it/s]

  Processing 82750 items from train/val...


Method C train/val:   0%|          | 0/9 [00:00<?, ?it/s]

💾 Saved 251008 embeddings to /kaggle/working/combined_embeddings/method_C.npz


## CheckpointManager

In [17]:
class CheckpointManager:
    def __init__(self, save_dir, task_name="compatibility"):
        self.save_dir = Path(save_dir) / task_name
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.task_name = task_name
        self.history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': [], 'learning_rates': [], 'epoch_times': []}
        self.best_metric = -float('inf')
        self.total_train_time = 0.0

    def epoch_start(self):
        self._epoch_start = time.time()

    def epoch_end(self, train_loss, val_loss, train_acc, val_acc, lr, epoch):
        epoch_time = time.time() - self._epoch_start
        self.total_train_time += epoch_time
        
        self.history['train_loss'].append(float(train_loss))
        self.history['val_loss'].append(float(val_loss))
        self.history['train_accuracy'].append(float(train_acc))
        self.history['val_accuracy'].append(float(val_acc))
        self.history['learning_rates'].append(float(lr))
        self.history['epoch_times'].append(float(epoch_time))
        
        is_best = val_acc > self.best_metric
        if is_best:
            self.best_metric = float(val_acc)
            
        print(f"  Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {lr:.2e} | Time: {epoch_time:.1f}s {' ★ BEST' if is_best else ''}")
        return is_best

    def save(self, model, optimizer, scheduler, epoch, is_best=False):
        checkpoint = {
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict() if scheduler else None,
            'epoch': epoch,
            'best_metric': self.best_metric,
            'history': self.history,
        }
        torch.save(checkpoint, self.save_dir / "checkpoint_latest.pth")
        if is_best:
            torch.save(checkpoint, self.save_dir / "checkpoint_best.pth")

## Training Helpers

In [18]:
from src.data import collate_fn
from src.data.datasets import polyvore
from src.evaluation.metrics import compute_cp_scores, compute_cir_scores
from src.models.load import load_model
from src.utils.loss import FocalLoss, InBatchTripletMarginLoss
from torch.utils.data import DataLoader

def setup_model(checkpoint_path):
    model = load_model('clip', checkpoint=checkpoint_path, image_only=False)
    
    # Freeze everything
    for param in model.parameters():
        param.requires_grad = False
        
    # Unfreeze prediction heads
    for name, param in model.named_parameters():
        if "predict_ffn" in name or "embed_ffn" in name:
            param.requires_grad = True
            
    # Unfreeze last 2 transformer layers
    max_layer_idx = -1
    for name, _ in model.named_parameters():
        if "style_enc.layers." in name:
            idx = int(name.split("style_enc.layers.")[1].split(".")[0])
            max_layer_idx = max(max_layer_idx, idx)
    if max_layer_idx >= 1:
        for name, param in model.named_parameters():
            if f"style_enc.layers.{max_layer_idx}." in name or f"style_enc.layers.{max_layer_idx - 1}." in name:
                param.requires_grad = True

    return model

def get_param_groups(model, base_lr):
    head_params, transformer_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "predict_ffn" in name or "embed_ffn" in name:
            head_params.append(param)
        else:
            transformer_params.append(param)
    return [
        {"params": head_params, "lr": base_lr},
        {"params": transformer_params, "lr": base_lr * 0.1},
    ]

README.md: 0.00B [00:00, ?B/s]

data/data-00000-of-00005.parquet:   0%|          | 0.00/369M [00:00<?, ?B/s]

data/data-00001-of-00005.parquet:   0%|          | 0.00/368M [00:00<?, ?B/s]

data/data-00002-of-00005.parquet:   0%|          | 0.00/369M [00:00<?, ?B/s]

data/data-00003-of-00005.parquet:   0%|          | 0.00/368M [00:00<?, ?B/s]

data/data-00004-of-00005.parquet:   0%|          | 0.00/368M [00:00<?, ?B/s]

data/data-00005-of-00005.parquet:   0%|          | 0.00/367M [00:00<?, ?B/s]

Generating data split: 0 examples [00:00, ? examples/s]

251008it [02:25, 1730.26it/s]


## Training Loops

In [19]:

def train_method(method_name, npz_path):
    print("\n" + "="*60)
    print(f"TRAINING METHOD: {method_name}")
    print("="*60)
    
    emb_dict = polyvore.load_embedding_dict(dataset_dir=None, npz_path=str(npz_path))
    ckpt_dir = Path(f"/kaggle/working/checkpoints/method_{method_name}")
    if not ckpt_dir.parent.exists():
         ckpt_dir = Path(f"../checkpoints/method_{method_name}") # local fallback
    
    # ---- 1. COMPATIBILITY ----
    print(f"--- 1. Compatibility ({method_name}) ---")
    c_train = polyvore.PolyvoreCompatibilityDataset(dataset_type='disjoint', dataset_split='train', embedding_dict=emb_dict)
    c_valid = polyvore.PolyvoreCompatibilityDataset(dataset_type='disjoint', dataset_split='validation', embedding_dict=emb_dict)
    c_train_loader = DataLoader(c_train, batch_size=512, shuffle=True, num_workers=0, collate_fn=collate_fn.cp_collate_fn)
    c_valid_loader = DataLoader(c_valid, batch_size=512, shuffle=False, num_workers=0, collate_fn=collate_fn.cp_collate_fn)
    
    model = setup_model('checkpoints/compatibillity_clip_best.pth')
    device = next(model.parameters()).device
    
    optimizer = torch.optim.AdamW(get_param_groups(model, 1e-3))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, epochs=10, steps_per_epoch=max(1, len(c_train_loader)//4), pct_start=0.3, anneal_strategy='cos')
    loss_fn = FocalLoss(alpha=0.5, gamma=2)
    ckpt = CheckpointManager(ckpt_dir, "compatibility")
    
    for epoch in range(10):
        ckpt.epoch_start()
        model.train()
        train_loss = 0.0
        preds_list, labels_list = [], []
        
        for i, data in enumerate(c_train_loader):
            labels = torch.tensor([float(l) for l in data['label']], dtype=torch.float32).to(device)
            preds = model(data['query'], use_precomputed_embedding=True).squeeze(1)
            loss = loss_fn(y_true=labels, y_prob=preds) / 4
            loss.backward()
            
            if (i + 1) % 4 == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
                
            train_loss += loss.item() * 4 / len(c_train_loader)
            preds_list.append(preds.detach())
            labels_list.append(labels.detach())
            
        train_scores = compute_cp_scores(torch.cat(preds_list), torch.cat(labels_list))
        
        model.eval()
        val_loss = 0.0
        preds_list, labels_list = [], []
        with torch.no_grad():
            for data in c_valid_loader:
                labels = torch.tensor([float(l) for l in data['label']], dtype=torch.float32).to(device)
                preds = model(data['query'], use_precomputed_embedding=True).squeeze(1)
                loss = loss_fn(y_true=labels, y_prob=preds)
                val_loss += loss.item() / len(c_valid_loader)
                preds_list.append(preds)
                labels_list.append(labels)
                
        val_scores = compute_cp_scores(torch.cat(preds_list), torch.cat(labels_list))
        
        is_best = ckpt.epoch_end(train_loss, val_loss, train_scores.get('auc', 0), val_scores.get('auc', 0), scheduler.get_last_lr()[0], epoch)
        ckpt.save(model, optimizer, scheduler, epoch, is_best)
        
    # ---- 2. COMPLEMENTARY ----
    print(f"\n--- 2. Complementary ({method_name}) ---")
    t_train = polyvore.PolyvoreTripletDataset(dataset_type='disjoint', dataset_split='train', embedding_dict=emb_dict)
    t_valid = polyvore.PolyvoreFillInTheBlankDataset(dataset_type='disjoint', dataset_split='validation', embedding_dict=emb_dict)
    t_train_loader = DataLoader(t_train, batch_size=64, shuffle=True, num_workers=0, collate_fn=collate_fn.triplet_collate_fn)
    t_valid_loader = DataLoader(t_valid, batch_size=64, shuffle=False, num_workers=0, collate_fn=collate_fn.fitb_collate_fn)
    
    model = setup_model('checkpoints/complementary_clip_best.pth')
    optimizer = torch.optim.AdamW(get_param_groups(model, 3e-4))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
    loss_fn = InBatchTripletMarginLoss(margin=0.5, reduction='mean')
    ckpt = CheckpointManager(ckpt_dir, "complementary")
    
    for epoch in range(10):
        ckpt.epoch_start()
        model.train()
        train_loss = 0.0
        preds_list, labels_list = [], []
        
        for i, data in enumerate(t_train_loader):
            q_emb = model(data['query'], use_precomputed_embedding=True)
            a_emb = model(data['answer'], use_precomputed_embedding=True)
            loss = loss_fn(q_emb, a_emb) / 4
            loss.backward()
            
            if (i + 1) % 4 == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
                
            dists = torch.cdist(q_emb.detach(), a_emb.detach(), p=2)
            preds = torch.argmin(dists, dim=1)
            labels = torch.arange(len(preds), device=device)
            
            train_loss += loss.item() * 4 / len(t_train_loader)
            preds_list.append(preds)
            labels_list.append(labels)
            
        train_scores = compute_cir_scores(torch.cat(preds_list), torch.cat(labels_list))
        
        model.eval()
        preds_list, labels_list = [], []
        with torch.no_grad():
            for data in t_valid_loader:
                q_emb = model(data['query'], use_precomputed_embedding=True).unsqueeze(1)
                c_embs = model(sum(data['candidates'], []), use_precomputed_embedding=True)
                c_embs = c_embs.view(-1, 4, c_embs.shape[1])
                dists = torch.norm(q_emb - c_embs, dim=-1)
                preds = torch.argmin(dists, dim=-1)
                labels = torch.tensor([int(l) for l in data['label']], device=device)
                preds_list.append(preds)
                labels_list.append(labels)
                
        val_scores = compute_cir_scores(torch.cat(preds_list), torch.cat(labels_list))
        
        is_best = ckpt.epoch_end(train_loss, 0.0, train_scores.get('acc', 0), val_scores.get('acc', 0), scheduler.get_last_lr()[0], epoch)
        scheduler.step()
        ckpt.save(model, optimizer, scheduler, epoch, is_best)


In [20]:
train_method("A", npz_A)


TRAINING METHOD: A
Loading embeddings from NPZ: /kaggle/working/combined_embeddings/method_A.npz
--- 1. Compatibility (A) ---


README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

valid.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.w

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.lay

  Epoch   1 | Train Loss: 0.9629 | Val Loss: 0.0000 | Train Acc: 0.0955 | Val Acc: 0.5047 | LR: 3.00e-04 | Time: 108.0s  ★ BEST
  Epoch   2 | Train Loss: 0.8167 | Val Loss: 0.0000 | Train Acc: 0.1083 | Val Acc: 0.5380 | LR: 2.93e-04 | Time: 108.1s  ★ BEST
  Epoch   3 | Train Loss: 0.7885 | Val Loss: 0.0000 | Train Acc: 0.1109 | Val Acc: 0.5280 | LR: 2.71e-04 | Time: 107.6s 
  Epoch   4 | Train Loss: 0.7446 | Val Loss: 0.0000 | Train Acc: 0.1164 | Val Acc: 0.5233 | LR: 2.38e-04 | Time: 107.1s 
  Epoch   5 | Train Loss: 0.7077 | Val Loss: 0.0000 | Train Acc: 0.1163 | Val Acc: 0.5440 | LR: 1.97e-04 | Time: 107.6s  ★ BEST
  Epoch   6 | Train Loss: 0.6745 | Val Loss: 0.0000 | Train Acc: 0.1260 | Val Acc: 0.5643 | LR: 1.50e-04 | Time: 106.7s  ★ BEST
  Epoch   7 | Train Loss: 0.6456 | Val Loss: 0.0000 | Train Acc: 0.1347 | Val Acc: 0.5623 | LR: 1.04e-04 | Time: 107.9s 
  Epoch   8 | Train Loss: 0.6284 | Val Loss: 0.0000 | Train Acc: 0.1382 | Val Acc: 0.5927 | LR: 6.26e-05 | Time: 107.2s  ★ BE

In [21]:
train_method("B", npz_B)


TRAINING METHOD: B
Loading embeddings from NPZ: /kaggle/working/combined_embeddings/method_B.npz
--- 1. Compatibility (B) ---


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.w

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.lay

...

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.w

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.lay

  Epoch   1 | Train Loss: 0.9548 | Val Loss: 0.0000 | Train Acc: 0.0964 | Val Acc: 0.4810 | LR: 3.00e-04 | Time: 109.3s  ★ BEST
  Epoch   2 | Train Loss: 0.8045 | Val Loss: 0.0000 | Train Acc: 0.1134 | Val Acc: 0.5017 | LR: 2.93e-04 | Time: 108.4s  ★ BEST
  Epoch   3 | Train Loss: 0.7651 | Val Loss: 0.0000 | Train Acc: 0.1170 | Val Acc: 0.5223 | LR: 2.71e-04 | Time: 108.0s  ★ BEST
  Epoch   4 | Train Loss: 0.7245 | Val Loss: 0.0000 | Train Acc: 0.1208 | Val Acc: 0.5137 | LR: 2.38e-04 | Time: 107.9s 
  Epoch   5 | Train Loss: 0.6991 | Val Loss: 0.0000 | Train Acc: 0.1260 | Val Acc: 0.5497 | LR: 1.97e-04 | Time: 109.1s  ★ BEST
  Epoch   6 | Train Loss: 0.6597 | Val Loss: 0.0000 | Train Acc: 0.1387 | Val Acc: 0.5463 | LR: 1.50e-04 | Time: 107.4s 
  Epoch   7 | Train Loss: 0.6385 | Val Loss: 0.0000 | Train Acc: 0.1398 | Val Acc: 0.5943 | LR: 1.04e-04 | Time: 110.7s  ★ BEST
  Epoch   8 | Train Loss: 0.6238 | Val Loss: 0.0000 | Train Acc: 0.1463 | Val Acc: 0.6093 | LR: 6.26e-05 | Time: 109.2

In [22]:
train_method("C", npz_C)


TRAINING METHOD: C
Loading embeddings from NPZ: /kaggle/working/combined_embeddings/method_C.npz
--- 1. Compatibility (C) ---


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.w

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.lay

...

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.w

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.lay

  Epoch   1 | Train Loss: 0.9786 | Val Loss: 0.0000 | Train Acc: 0.0963 | Val Acc: 0.5337 | LR: 3.00e-04 | Time: 108.3s  ★ BEST
  Epoch   2 | Train Loss: 0.8399 | Val Loss: 0.0000 | Train Acc: 0.1114 | Val Acc: 0.5133 | LR: 2.93e-04 | Time: 108.4s 
  Epoch   3 | Train Loss: 0.7974 | Val Loss: 0.0000 | Train Acc: 0.1114 | Val Acc: 0.5500 | LR: 2.71e-04 | Time: 106.4s  ★ BEST
  Epoch   4 | Train Loss: 0.7527 | Val Loss: 0.0000 | Train Acc: 0.1139 | Val Acc: 0.5490 | LR: 2.38e-04 | Time: 105.9s 
  Epoch   5 | Train Loss: 0.7158 | Val Loss: 0.0000 | Train Acc: 0.1192 | Val Acc: 0.5553 | LR: 1.97e-04 | Time: 107.2s  ★ BEST
  Epoch   6 | Train Loss: 0.6884 | Val Loss: 0.0000 | Train Acc: 0.1216 | Val Acc: 0.5957 | LR: 1.50e-04 | Time: 108.3s  ★ BEST
  Epoch   7 | Train Loss: 0.6565 | Val Loss: 0.0000 | Train Acc: 0.1276 | Val Acc: 0.5823 | LR: 1.04e-04 | Time: 108.3s 
  Epoch   8 | Train Loss: 0.6394 | Val Loss: 0.0000 | Train Acc: 0.1370 | Val Acc: 0.5933 | LR: 6.26e-05 | Time: 108.3s 
  Ep

## Evaluation & Baseline Comparison

In [23]:

def eval_method(method_name, npz_path, ckpt_dir):
    print(f"Evaluating {method_name}...")
    emb_dict = polyvore.load_embedding_dict(dataset_dir=None, npz_path=str(npz_path))
    
    # 1. Compatibility (CP)
    c_test = polyvore.PolyvoreCompatibilityDataset(dataset_type='disjoint', dataset_split='test', embedding_dict=emb_dict)
    c_loader = DataLoader(c_test, batch_size=512, shuffle=False, num_workers=0, collate_fn=collate_fn.cp_collate_fn)
    
    c_ckpt = torch.load(ckpt_dir / "compatibility" / "checkpoint_best.pth", map_location='cpu')
    c_model = load_model('clip', image_only=False)
    c_model.load_state_dict({k.replace("module.", ""): v for k, v in c_ckpt['model'].items()}, strict=False)
    c_model.eval()
    device = next(c_model.parameters()).device
    
    preds_list, labels_list = [], []
    with torch.no_grad():
        for data in c_loader:
            labels = torch.tensor([float(l) for l in data['label']], dtype=torch.float32).to(device)
            preds = c_model(data['query'], use_precomputed_embedding=True).squeeze(1)
            preds_list.append(preds)
            labels_list.append(labels)
    cp_scores = compute_cp_scores(torch.cat(preds_list), torch.cat(labels_list))
    
    # 2. Complementary (CIR)
    t_test = polyvore.PolyvoreFillInTheBlankDataset(dataset_type='disjoint', dataset_split='test', embedding_dict=emb_dict)
    t_loader = DataLoader(t_test, batch_size=64, shuffle=False, num_workers=0, collate_fn=collate_fn.fitb_collate_fn)
    
    t_ckpt = torch.load(ckpt_dir / "complementary" / "checkpoint_best.pth", map_location='cpu')
    t_model = load_model('clip', image_only=False)
    t_model.load_state_dict({k.replace("module.", ""): v for k, v in t_ckpt['model'].items()}, strict=False)
    t_model.eval()
    
    preds_list, labels_list = [], []
    with torch.no_grad():
        for data in t_loader:
            q_emb = t_model(data['query'], use_precomputed_embedding=True).unsqueeze(1)
            c_embs = t_model(sum(data['candidates'], []), use_precomputed_embedding=True)
            c_embs = c_embs.view(-1, 4, c_embs.shape[1])
            dists = torch.norm(q_emb - c_embs, dim=-1)
            preds = torch.argmin(dists, dim=-1)
            labels = torch.tensor([int(l) for l in data['label']], device=device)
            preds_list.append(preds)
            labels_list.append(labels)
    cir_scores = compute_cir_scores(torch.cat(preds_list), torch.cat(labels_list))
    
    return {"cp": cp_scores, "cir": cir_scores}

In [24]:
def eval_baseline():
    print(f"Evaluating Baseline Image-Only...")
    
    # Needs to load old embeddings directory instead of NPZ
    # For baseline, the embedding dict was read from `.npy` files.
    emb_dict = polyvore.load_embedding_dict(str(embeddings_dir)) 
    
    # 1. Compatibility
    c_test = polyvore.PolyvoreCompatibilityDataset(dataset_type='disjoint', dataset_split='test', embedding_dict=emb_dict)
    c_loader = DataLoader(c_test, batch_size=512, shuffle=False, num_workers=0, collate_fn=collate_fn.cp_collate_fn)
    
    c_model = load_model('clip', checkpoint='checkpoints/compatibillity_clip_best.pth', image_only=True)
    c_model.eval()
    device = next(c_model.parameters()).device
    
    preds_list, labels_list = [], []
    with torch.no_grad():
        for data in c_loader:
            labels = torch.tensor([float(l) for l in data['label']], dtype=torch.float32).to(device)
            preds = c_model(data['query'], use_precomputed_embedding=True).squeeze(1)
            preds_list.append(preds)
            labels_list.append(labels)
    cp_scores = compute_cp_scores(torch.cat(preds_list), torch.cat(labels_list))
    
    # 2. Complementary
    t_test = polyvore.PolyvoreFillInTheBlankDataset(dataset_type='disjoint', dataset_split='test', embedding_dict=emb_dict)
    t_loader = DataLoader(t_test, batch_size=64, shuffle=False, num_workers=0, collate_fn=collate_fn.fitb_collate_fn)
    
    t_model = load_model('clip', checkpoint='checkpoints/complementary_clip_best.pth', image_only=True)
    t_model.eval()
    
    preds_list, labels_list = [], []
    with torch.no_grad():
        for data in t_loader:
            q_emb = t_model(data['query'], use_precomputed_embedding=True).unsqueeze(1)
            c_embs = t_model(sum(data['candidates'], []), use_precomputed_embedding=True)
            c_embs = c_embs.view(-1, 4, c_embs.shape[1])
            dists = torch.norm(q_emb - c_embs, dim=-1)
            preds = torch.argmin(dists, dim=-1)
            labels = torch.tensor([int(l) for l in data['label']], device=device)
            preds_list.append(preds)
            labels_list.append(labels)
    cir_scores = compute_cir_scores(torch.cat(preds_list), torch.cat(labels_list))
    
    return {"cp": cp_scores, "cir": cir_scores}

In [25]:
results = {}
base_ckpt_dir = Path("/kaggle/working/checkpoints") if Path("/kaggle/working/checkpoints").exists() else Path("../checkpoints")

try:
    results["A (Top-1)"] = eval_method("A", npz_A, base_ckpt_dir / "method_A")
    results["B (Top-5 Avg)"] = eval_method("B", npz_B, base_ckpt_dir / "method_B")
    results["C (Dup Image)"] = eval_method("C", npz_C, base_ckpt_dir / "method_C")
    
    try:
        results["Baseline (Img-Only)"] = eval_baseline()
    except Exception as e:
        print(f"Warning: Baseline evaluation failed. Make sure 'checkpoints/compatibillity_clip_best.pth' has the image_only weights. ({e})")
except Exception as e:
    print(f"Evaluation failed: {e}")

Evaluating A...
Loading embeddings from NPZ: /kaggle/working/combined_embeddings/method_A.npz


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.w

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.lay

...

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.w

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: patrickjohncyh/fashion-clip
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.lay

Evaluating Baseline Image-Only...


## Print Final Table

In [26]:
print("\n" + "=" * 80)
print(f"{'FINAL TEST RESULTS COMPARISON':^80}")
print("=" * 80)
print(f"{'Method':<20} | {'CP AUC':<10} | {'CP Acc':<10} | {'CP F1':<10} | {'FITB Acc (CIR)':<15}")
print("-" * 80)

for method, res in results.items():
    cp_auc = res['cp'].get('auc', 0)
    cp_acc = res['cp'].get('acc', 0)
    cp_f1 = res['cp'].get('f1', 0)
    cir_acc = res['cir'].get('acc', 0)
    
    print(f"{method:<20} | {cp_auc:<10.4f} | {cp_acc:<10.4f} | {cp_f1:<10.4f} | {cir_acc:<15.4f}")

print("-" * 80)
print("Conclusion:")
print("Check the table above to determine the best method.")


                         FINAL TEST RESULTS COMPARISON                          
Method               | CP AUC     | CP Acc     | CP F1      | FITB Acc (CIR) 
--------------------------------------------------------------------------------
A (Top-1)            | 0.9054     | 0.8216     | 0.8166     | 0.6149         
B (Top-5 Avg)        | 0.9141     | 0.8296     | 0.8219     | 0.6126         
C (Dup Image)        | 0.9040     | 0.8169     | 0.8079     | 0.6172         
--------------------------------------------------------------------------------
Conclusion:
Check the table above to determine the best method.


In [27]:
# Install the library if you haven't already
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login
import os

# 1. Authenticate
# If in Kaggle, use Kaggle Secrets:
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret("HF_TOKEN")
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("HF_TOKEN") # Replace with your token if running locally

login(token=token)

# 2. Initialize the API
api = HfApi()

# 3. Define Repository Details
# Format must be "your-hf-username/repo-name"
repo_id = "FitFinder/outfit-transformer-checkpoints" 

# 4. Create the Repository
print(f"Creating repository: {repo_id}...")
api.create_repo(
    repo_id=repo_id,
    repo_type="model", # Use "model" for weights, or "dataset" for pure data
    private=True,      # Set to False if you want it public
    exist_ok=True      # Won't throw an error if the repo already exists
)
print("✅ Repository ready!")

# 5. Upload your Checkpoints/Results Folder
folder_to_upload = "/kaggle/working/checkpoints" # Update to your actual folder path

print(f"Uploading contents of {folder_to_upload}...")
api.upload_folder(
    folder_path=folder_to_upload,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Adding latest training checkpoints and results"
)
print("🚀 Upload complete!")

Creating repository: FitFinder/outfit-transformer-checkpoints...
✅ Repository ready!
Uploading contents of /kaggle/working/checkpoints...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

🚀 Upload complete!
